In [ ]:
# ==============================================================================
# PUMPKIN LEAF DISEASE: LOCALIZATION, SEGMENTATION & SEVERITY ESTIMATION PIPELINE
# Follows Chapter Methodology (Phase 0 to Phase 7)
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/pumpkin_data'):
    !unzip -q "/content/drive/MyDrive/Original Dataset.zip" -d /content/pumpkin_data


In [ ]:
import os, sys, hashlib, json, time, random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

OUT_BASE = '/content/drive/MyDrive/pumpkin_severity_runs'
os.makedirs(OUT_BASE, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | Output Directory: {OUT_BASE}")


In [ ]:
# ==============================================================================
# PHASE 0: SET UP & DUPLICATE AUDIT
# - Standardize IDs (e.g., BLS_0001, DM_0001, HL_0001, MD_0001, PM_0001)
# - Compute MD5 file hashes to detect exact duplicates
# - Record resolution and file metadata
# ==============================================================================
def find_root(base):
    for r, dirs, _ in os.walk(base):
        if len(dirs) == 5 and all(any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in os.listdir(os.path.join(r, d))) for d in dirs):
            return r
DATA_ROOT = find_root('/content/pumpkin_data')

CLASS_PREFIX = {
    'Bacterial Leaf Spot': 'BLS',
    'Downy Mildew': 'DM',
    'Healthy Leaf': 'HL',
    'Mosaic Disease': 'MD',
    'Powdery_Mildew': 'PM'
}

master_records = []
classes = sorted(os.listdir(DATA_ROOT))

for cls in classes:
    cls_folder = os.path.join(DATA_ROOT, cls)
    if not os.path.isdir(cls_folder): continue
    files = sorted([f for f in os.listdir(cls_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    prefix = CLASS_PREFIX.get(cls, cls[:3].upper())
    
    for idx, fname in enumerate(files):
        fpath = os.path.join(cls_folder, fname)
        rel_path = os.path.relpath(fpath, DATA_ROOT)
        img_id = f"{prefix}_{idx+1:04d}"
        
        # File MD5 hash for exact duplicate detection
        hasher = hashlib.md5()
        with open(fpath, 'rb') as f:
            hasher.update(f.read())
        fhash = hasher.hexdigest()
        
        # Image dimensions
        with Image.open(fpath) as img:
            w, h = img.size
            mode = img.mode
            
        master_records.append({
            'image_id': img_id,
            'class': cls,
            'filename': fname,
            'rel_path': rel_path,
            'abs_path': fpath,
            'width': w,
            'height': h,
            'channels': mode,
            'md5_hash': fhash
        })

df_master = pd.DataFrame(master_records)
master_csv = os.path.join(OUT_BASE, 'master_dataset_index.csv')
df_master.to_csv(master_csv, index=False)
print(f"Master index saved: {len(df_master)} images across {len(classes)} classes.")

# Check for exact duplicate file hashes
duplicates = df_master[df_master.duplicated(subset=['md5_hash'], keep=False)]
print(f"Exact duplicates detected: {len(duplicates)}")
if len(duplicates) > 0:
    print(duplicates[['image_id', 'class', 'filename', 'md5_hash']].head())


In [ ]:
# ==============================================================================
# PHASE 1: DATASET AUDIT SAMPLING (200 images: 40 per class)
# Generates the audit spreadsheet for human evaluation of field clutter
# ==============================================================================
AUDIT_SAMPLE_PER_CLASS = 40
audit_samples = []

for cls in classes:
    cls_df = df_master[df_master['class'] == cls]
    sampled = cls_df.sample(n=min(AUDIT_SAMPLE_PER_CLASS, len(cls_df)), random_state=SEED)
    audit_samples.append(sampled)

df_audit = pd.concat(audit_samples).reset_index(drop=True)

# Add standardized audit survey columns
audit_columns = [
    'num_leaves_visible',      # Integer: count of leaves visible in frame
    'focal_leaf_obvious',      # 1 (Yes) / 0 (No)
    'healthy_neighbours',      # 1 (Yes, healthy foliage present) / 0 (No)
    'leaf_boundary_clarity',   # 2: clear, 1: partly clear, 0: unclear
    'occlusion_grass_stems',   # 0: none, 1: some, 2: heavy
    'dry_dead_tissue_present', # 1 (Yes) / 0 (No)
    'holes_present',           # 1 (Yes) / 0 (No)
    'diffuse_yellowing',       # 1 (Yes, chlorosis without distinct margins) / 0 (No)
    'lesion_boundary_sharp',   # 2: sharp, 1: diffuse/vague, 0: no lesions (healthy)
    'lighting_quality'         # 2: good/even, 1: harsh shadow/glare, 0: underexposed/blurred
]

for col in audit_columns:
    df_audit[col] = ""

audit_csv = os.path.join(OUT_BASE, 'dataset_audit_sheet_200.csv')
df_audit.to_csv(audit_csv, index=False)
print(f"Audit sheet generated: {audit_csv} ({len(df_audit)} images: 40 per class)")
print("Fill this CSV during the visual audit. The next cell computes decision metrics.")


In [ ]:
# ==============================================================================
# PHASE 1 & 5: AUDIT SUMMARY & DETECTION DECISION RULE
# Objective: Statistically verify if a Detection stage (RT-DETR / YOLO) is required
# ==============================================================================
def evaluate_audit_decision(audit_csv_path, multi_leaf_threshold=0.25):
    if not os.path.exists(audit_csv_path):
        print(f"Audit file not yet filled: {audit_csv_path}")
        return
        
    df = pd.read_csv(audit_csv_path)
    # Check if filled
    if df['num_leaves_visible'].isna().all() or (df['num_leaves_visible'] == "").all():
        print("Audit sheet is not yet filled. Please fill the survey columns first.")
        return
        
    df['multi_leaf'] = pd.to_numeric(df['num_leaves_visible'], errors='coerce') > 1
    df['healthy_neighbours'] = pd.to_numeric(df['healthy_neighbours'], errors='coerce') == 1
    
    summary = df.groupby('class').agg(
        total_images=('image_id', 'count'),
        multi_leaf_pct=('multi_leaf', lambda x: np.mean(x) * 100),
        healthy_neighbour_pct=('healthy_neighbours', lambda x: np.mean(x) * 100)
    ).reset_index()
    
    print("=== DATASET AUDIT SUMMARY ===")
    print(summary.to_string(index=False))
    
    overall_multi_leaf = df['multi_leaf'].mean()
    print(f"\nOverall Multi-Leaf Occurrence: {overall_multi_leaf*100:.1f}% (Threshold: {multi_leaf_threshold*100:.1f}%)")
    
    if overall_multi_leaf > multi_leaf_threshold:
        print("[DECISION RULE]: Multi-leaf/neighbour presence exceeds threshold.")
        print("==> Stage 1 Target-Leaf Detection (YOLO/RT-DETR) is AGRO-ECONOMICALLY JUSTIFIED.")
    else:
        print("[DECISION RULE]: Isolated single-leaf frames predominate.")
        print("==> Direct whole-image Segmentation without preliminary detection is sufficient.")

evaluate_audit_decision(audit_csv)


In [ ]:
# ==============================================================================
# PHASE 2 & 3: PILOT ANNOTATION PARSER & INTER-RATER AGREEMENT (DICE)
# Compares masks from two independent annotators on the 20-image sanity set
# ==============================================================================
def compute_mask_dice(mask_a, mask_b):
    intersection = np.logical_and(mask_a, mask_b).sum()
    total = mask_a.sum() + mask_b.sum()
    if total == 0:
        return 1.0  # Perfect agreement on empty masks (e.g. Healthy lesion masks)
    return 2.0 * intersection / total

def evaluate_inter_rater_reliability(rater1_dir, rater2_dir, sample_ids):
    dice_leaf_scores, dice_lesion_scores, severity_deltas = [], [], []
    
    for sid in sample_ids:
        # Expected mask filenames: {sid}_leaf.png and {sid}_lesion.png
        f_leaf1 = os.path.join(rater1_dir, f"{sid}_leaf.png")
        f_leaf2 = os.path.join(rater2_dir, f"{sid}_leaf.png")
        f_les1  = os.path.join(rater1_dir, f"{sid}_lesion.png")
        f_les2  = os.path.join(rater2_dir, f"{sid}_lesion.png")
        
        if not (os.path.exists(f_leaf1) and os.path.exists(f_leaf2)):
            continue
            
        leaf1 = np.array(Image.open(f_leaf1).convert('L')) > 0
        leaf2 = np.array(Image.open(f_leaf2).convert('L')) > 0
        les1  = (np.array(Image.open(f_les1).convert('L')) > 0) & leaf1 if os.path.exists(f_les1) else np.zeros_like(leaf1)
        les2  = (np.array(Image.open(f_les2).convert('L')) > 0) & leaf2 if os.path.exists(f_les2) else np.zeros_like(leaf2)
        
        # Dice calculation
        dice_leaf = compute_mask_dice(leaf1, leaf2)
        dice_les = compute_mask_dice(les1, les2)
        
        # Severity calculation
        sev1 = 100.0 * les1.sum() / max(1, leaf1.sum())
        sev2 = 100.0 * les2.sum() / max(1, leaf2.sum())
        delta_sev = abs(sev1 - sev2)
        
        dice_leaf_scores.append(dice_leaf)
        dice_lesion_scores.append(dice_les)
        severity_deltas.append(delta_sev)
        
    if dice_leaf_scores:
        print("=== INTER-RATER ANNOTATION AUDIT (20 Pilot Images) ===")
        print(f"Mean Leaf Mask Dice:        {np.mean(dice_leaf_scores):.4f}")
        print(f"Mean Lesion Mask Dice:      {np.mean(dice_lesion_scores):.4f}")
        print(f"Mean Absolute Severity Diff:{np.mean(severity_deltas):.2f}%")
    else:
        print("Pilot masks not yet placed in rater directories. Ready for evaluation upon annotation completion.")


In [ ]:
# ==============================================================================
# PHASE 4: QUANTITATIVE SEVERITY ENGINE & VISUALIZER
# Formula: Severity % = (Lesion Pixels within Target Leaf / Total Target Leaf Pixels) * 100
# ==============================================================================
def calculate_and_visualize_severity(img_path, leaf_mask_path, lesion_mask_path, save_path=None):
    img = Image.open(img_path).convert('RGB')
    leaf = np.array(Image.open(leaf_mask_path).convert('L')) > 0
    lesion = np.array(Image.open(lesion_mask_path).convert('L')) > 0
    
    # Critical agronomical rule: Lesion pixels outside the target leaf denominator are pruned
    valid_lesion = lesion & leaf
    leaf_pixel_count = leaf.sum()
    lesion_pixel_count = valid_lesion.sum()
    
    severity_pct = (lesion_pixel_count / leaf_pixel_count * 100.0) if leaf_pixel_count > 0 else 0.0
    
    # 4-Panel Plot
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img); axes[0].set_title("Input Image"); axes[0].axis('off')
    axes[1].imshow(leaf, cmap='Greens'); axes[1].set_title(f"Target Leaf Mask\n({leaf_pixel_count:,} px)"); axes[1].axis('off')
    axes[2].imshow(valid_lesion, cmap='Reds'); axes[2].set_title(f"Lesion Mask\n({lesion_pixel_count:,} px)"); axes[2].axis('off')
    
    # Overlay on original image
    overlay = np.array(img).copy()
    overlay[leaf] = (overlay[leaf] * 0.7 + np.array([0, 255, 0]) * 0.3).astype(np.uint8)
    overlay[valid_lesion] = (overlay[valid_lesion] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
    axes[3].imshow(overlay)
    axes[3].set_title(f"Severity: {severity_pct:.2f}%")
    axes[3].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    return severity_pct


In [ ]:
# ==============================================================================
# PHASE 5: AUTO-EXTRACT BOUNDING BOXES FOR TARGET-LEAF DETECTOR (YOLO/RT-DETR)
# Converts binary leaf masks into normalized YOLO format [class x_center y_center w h]
# ==============================================================================
def mask_to_yolo_bbox(mask_array):
    rows = np.any(mask_array, axis=1)
    cols = np.any(mask_array, axis=0)
    if not np.any(rows) or not np.any(cols):
        return None
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]
    
    h, w = mask_array.shape
    x_center = ((xmin + xmax) / 2.0) / w
    y_center = ((ymin + ymax) / 2.0) / h
    box_w = (xmax - xmin) / w
    box_h = (ymax - ymin) / h
    return (0, x_center, y_center, box_w, box_h)  # Class 0: target_leaf


In [ ]:
# ==============================================================================
# PHASE 6: SEGMENTATION BASELINE MODEL
# Two-Head or Dual-Model U-Net for Leaf Mask and Lesion Mask Prediction
# Uses Dice-Focal Loss to counter extreme pixel imbalance
# ==============================================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class DualTaskUNet(nn.Module):
    """
    Predicts 2 binary mask channels:
      Channel 0: Leaf Mask (Target leaf denominator)
      Channel 1: Lesion Mask (Diseased lesions numerator)
    """
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        
        self.out_conv = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        b = self.bottleneck(self.pool(e4))
        
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)

class DiceFocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = targets * probs + (1 - targets) * (1 - probs)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * bce
        
        # Dice Loss
        intersection = (probs * targets).sum(dim=(2, 3))
        dice = (2.0 * intersection + self.smooth) / (probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + self.smooth)
        dice_loss = 1.0 - dice
        
        return focal_loss.mean() + dice_loss.mean()

seg_model = DualTaskUNet(in_channels=3, out_channels=2).to(device)
print(f"DualTaskUNet initialized with {sum(p.numel() for p in seg_model.parameters())/1e6:.2f}M parameters")


In [ ]:
# ==============================================================================
# PHASE 6: EVALUATION METRICS (IoU, Dice, Severity MAE & R2)
# Strictly separates mask quality (IoU/Dice) from severity error (MAE/R2)
# ==============================================================================
from sklearn.metrics import mean_absolute_error, r2_score

def evaluate_segmentation_and_severity(y_true_leaves, y_pred_leaves, y_true_lesions, y_pred_lesions):
    """
    Computes:
    - Leaf IoU & Dice
    - Lesion IoU & Dice
    - Severity MAE & R2
    """
    leaf_ious, leaf_dices = [], []
    lesion_ious, lesion_dices = [], []
    gt_severities, pred_severities = [], []
    
    for l_gt, l_pr, s_gt, s_pr in zip(y_true_leaves, y_pred_leaves, y_true_lesions, y_pred_lesions):
        # Leaf metrics
        inter_l = np.logical_and(l_gt, l_pr).sum()
        union_l = np.logical_or(l_gt, l_pr).sum()
        leaf_ious.append(inter_l / max(1, union_l))
        leaf_dices.append(2.0 * inter_l / max(1, l_gt.sum() + l_pr.sum()))
        
        # Lesion metrics (masked within respective leaves)
        s_gt_clean = s_gt & l_gt
        s_pr_clean = s_pr & l_pr
        inter_s = np.logical_and(s_gt_clean, s_pr_clean).sum()
        union_s = np.logical_or(s_gt_clean, s_pr_clean).sum()
        lesion_ious.append(inter_s / max(1, union_s))
        lesion_dices.append(2.0 * inter_s / max(1, s_gt_clean.sum() + s_pr_clean.sum()))
        
        # Severities
        sev_gt = 100.0 * s_gt_clean.sum() / max(1, l_gt.sum())
        sev_pr = 100.0 * s_pr_clean.sum() / max(1, l_pr.sum())
        gt_severities.append(sev_gt)
        pred_severities.append(sev_pr)
        
    mae = mean_absolute_error(gt_severities, pred_severities)
    r2 = r2_score(gt_severities, pred_severities) if len(gt_severities) > 1 else 0.0
    
    print("=== SEGMENTATION & SEVERITY EVALUATION ===")
    print(f"Leaf Mask    | Mean IoU: {np.mean(leaf_ious)*100:.2f}% | Mean Dice: {np.mean(leaf_dices):.4f}")
    print(f"Lesion Mask  | Mean IoU: {np.mean(lesion_ious)*100:.2f}% | Mean Dice: {np.mean(lesion_dices):.4f}")
    print(f"Severity     | MAE: {mae:.2f}% | R²: {r2:.4f}")


In [ ]:
# ==============================================================================
# PHASE 7: HEATMAP OVERLAP AS INTERPRETABILITY LAYER (NOT SEVERITY MEASURE)
# Evaluates Pointwise Agreement & IoU between Grad-CAM activation and Reference Lesion Masks
# ==============================================================================
def evaluate_heatmap_lesion_overlap(cam_heatmap, gt_lesion_mask, threshold=0.5):
    """
    cam_heatmap: 2D float array normalized between [0, 1]
    gt_lesion_mask: 2D binary bool array
    """
    cam_binary = cam_heatmap >= threshold
    intersection = np.logical_and(cam_binary, gt_lesion_mask).sum()
    union = np.logical_or(cam_binary, gt_lesion_mask).sum()
    
    iou = intersection / max(1, union)
    pointwise_agreement = np.mean(cam_binary == gt_lesion_mask)
    return iou, pointwise_agreement


In [ ]:
# ==============================================================================
# PIPELINE STATUS & LOG
# ==============================================================================
pipeline_status = pd.DataFrame([
    {'Phase': 'Phase 0', 'Task': 'Master Dataset Index & Duplicate Hash Check', 'Status': 'Completed', 'Output': 'master_dataset_index.csv'},
    {'Phase': 'Phase 1', 'Task': 'Dataset Audit Sample (200 images, 40/class)', 'Status': 'Ready for scoring', 'Output': 'dataset_audit_sheet_200.csv'},
    {'Phase': 'Phase 2', 'Task': 'Annotation Specs & Two-Label Protocols', 'Status': 'Defined (Leaf & Lesion)', 'Output': 'Protocols documented'},
    {'Phase': 'Phase 3', 'Task': 'Pilot Annotation & Inter-rater Dice', 'Status': 'Ready for 20 sanity masks', 'Output': 'Dice & Severity delta'},
    {'Phase': 'Phase 4', 'Task': 'Severity Engine from Masks', 'Status': 'Completed', 'Output': '4-panel visualizer'},
    {'Phase': 'Phase 5', 'Task': 'Leaf Bounding Box Generator', 'Status': 'Completed', 'Output': 'YOLO/RT-DETR box format'},
    {'Phase': 'Phase 6', 'Task': 'DualTaskUNet & DiceFocalLoss', 'Status': 'Architecture Built', 'Output': 'IoU/Dice & MAE/R2'},
    {'Phase': 'Phase 7', 'Task': 'Grad-CAM vs Lesion Overlap', 'Status': 'Built', 'Output': 'Pointwise & IoU overlap'}
])
pipeline_status
